In [1]:
DATA_PATH = "pp-2025-part-*.csv"
MODEL_PATH = "artifacts/model.joblib"
METRICS_PATH = "artifacts/metrics.json"
IMPORTANCE_PATH = "artifacts/feature_importance.csv"
RANDOM_STATE = 42
COLUMNS = ["transaction_id", "price", "date", "postcode", "property_type", "old_new", "duration", "paon", "saon", "street", "locality", "town", "district", "county", "ppd_category", "record_status"]
FEATURES = ["property_type", "old_new", "duration", "district", "county", "outcode", "month"]
CAT_FEATURES = ["property_type", "old_new", "duration", "district", "county", "outcode"]


In [2]:
import json
from pathlib import Path
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor


In [3]:
data_files = sorted(Path(".").glob(DATA_PATH))
if not data_files:
    raise FileNotFoundError("No split dataset files found")
df = pd.concat(
    [pd.read_csv(path, header=None, names=COLUMNS, dtype=str) for path in data_files],
    ignore_index=True,
)
df["price"] = pd.to_numeric(df["price"], errors="coerce")
print(df.shape)
print(df.head(3))


(954145, 16)
                           transaction_id   price              date  postcode  \
0  {42C129E4-C259-60A9-E063-4804A8C0C25D}  635000  2025-05-14 00:00  LN11 8GN   
1  {42C129E4-C25B-60A9-E063-4804A8C0C25D}  230000  2025-08-29 00:00  PE10 2BL   
2  {42C129E4-C25E-60A9-E063-4804A8C0C25D}  185000  2025-03-27 00:00  LN10 6RJ   

  property_type old_new duration paon saon              street locality  \
0             D       N        F    2  NaN        KENWICK VIEW      NaN   
1             S       N        F    6  NaN        CONWAY DRIVE      NaN   
2             S       N        F    6  NaN  KING EDWARD AVENUE      NaN   

           town        district        county ppd_category record_status  
0         LOUTH    EAST LINDSEY  LINCOLNSHIRE            A             A  
1        BOURNE  SOUTH KESTEVEN  LINCOLNSHIRE            A             A  
2  WOODHALL SPA    EAST LINDSEY  LINCOLNSHIRE            A             A  


In [4]:
print(df.dtypes)
print(df.shape)
print(df.head())


transaction_id      str
price             int64
date                str
postcode            str
property_type       str
old_new             str
duration            str
paon                str
saon                str
street              str
locality            str
town                str
district            str
county              str
ppd_category        str
record_status       str
dtype: object
(954145, 16)
                           transaction_id   price              date  postcode  \
0  {42C129E4-C259-60A9-E063-4804A8C0C25D}  635000  2025-05-14 00:00  LN11 8GN   
1  {42C129E4-C25B-60A9-E063-4804A8C0C25D}  230000  2025-08-29 00:00  PE10 2BL   
2  {42C129E4-C25E-60A9-E063-4804A8C0C25D}  185000  2025-03-27 00:00  LN10 6RJ   
3  {42C129E4-C260-60A9-E063-4804A8C0C25D}   76000  2025-06-06 00:00  NG31 8SP   
4  {42C129E4-C261-60A9-E063-4804A8C0C25D}  237500  2025-09-18 00:00  PE11 4PP   

  property_type old_new duration paon saon              street   locality  \
0             D       N  

In [5]:
Path("artifacts").mkdir(exist_ok=True)
plt.figure()
plt.hist(np.log1p(df["price"].dropna()), bins=80)
plt.title("log1p price")
plt.savefig("artifacts/eda_price_hist.png")
plt.close()


In [6]:
df = df[df["ppd_category"] == "A"].copy()
print(df.shape)


(794895, 16)


In [7]:
df = df[df["price"] > 0]
print(df.shape)


(794895, 16)


In [8]:
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.month


In [9]:
df["outcode"] = df["postcode"].str.split().str[0]
df = df[df["outcode"].notna()]


In [10]:
df = df.drop_duplicates(subset=["transaction_id"] if df["transaction_id"].is_unique else ["postcode", "date", "price", "paon", "saon", "street"])
print(df.shape)
print(df[FEATURES].isna().sum())


(794697, 18)
property_type    0
old_new          0
duration         0
district         0
county           0
outcode          0
month            0
dtype: int64


In [11]:
X = df[FEATURES].copy()


In [12]:
split_date = pd.Timestamp("2025-10-01")
train_rows, test_rows = df["date"] <= split_date, df["date"] > split_date
cap = df.loc[train_rows, "price"].quantile(0.999)
train_rows &= df["price"] <= cap
X_train, X_test = X.loc[train_rows], X.loc[test_rows]
y_train, y_test = np.log1p(df.loc[train_rows, "price"]), np.log1p(df.loc[test_rows, "price"])
print(X_train.shape, X_test.shape)


(607585, 7) (186507, 7)


In [13]:
pipe = CatBoostRegressor(loss_function="RMSE", iterations=500, verbose=False, random_seed=RANDOM_STATE)
pipe.fit(X_train, y_train, cat_features=CAT_FEATURES)


CatBoostRegressor(iterations=500, loss_function='RMSE', random_seed=42, verbose=False)

In [14]:
y_pred_log = pipe.predict(X_test)
y_true = np.expm1(y_test)
y_pred = np.expm1(y_pred_log)


In [15]:
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
baseline = np.expm1(y_train.median())
mdape = np.median(np.abs((y_true - y_pred) / y_true)) * 100
print(mae, rmse, r2, mdape)


97036.93469137883 253291.7181513605 0.46657868405065117 18.845819548431844


In [16]:
sample_idx = np.random.default_rng(RANDOM_STATE).choice(len(y_true), size=min(5000, len(y_true)), replace=False)
plt.figure()
plt.scatter(y_true.iloc[sample_idx], y_pred[sample_idx], alpha=0.2, s=5)
plt.xlabel("actual")
plt.ylabel("predicted")
plt.savefig("artifacts/residuals.png")
plt.close()


In [17]:
perm_idx = X_test.sample(min(5000, len(X_test)), random_state=RANDOM_STATE).index
perm = permutation_importance(pipe, X_test.loc[perm_idx], y_test.loc[perm_idx], n_repeats=5, random_state=RANDOM_STATE)
importances = perm.importances_mean


In [18]:
importance_df = pd.DataFrame({"feature": FEATURES, "importance": importances}).sort_values("importance", ascending=False)
print(importance_df)


         feature  importance
0  property_type    0.450599
5        outcode    0.349747
3       district    0.172498
4         county    0.120133
2       duration    0.107045
1        old_new    0.001874
6          month    0.000000


In [19]:
top_n = len(FEATURES)
top = importance_df.head(top_n)
plt.figure(figsize=(8, 5))
plt.barh(top["feature"][::-1], top["importance"][::-1])
plt.xlabel("importance")
plt.title("top feature importances")
plt.tight_layout()
plt.savefig("artifacts/feature_importance.png")
plt.close()


In [20]:
Path("artifacts").mkdir(exist_ok=True)
joblib.dump(pipe, MODEL_PATH)
importance_df.to_csv(IMPORTANCE_PATH, index=False)


In [21]:
metrics = {"mae": mae, "rmse": rmse, "r2": r2, "mdape_percent": mdape, "baseline_median_mae": mean_absolute_error(y_true, np.full(len(y_true), baseline)), "n_train": len(X_train), "n_test": len(X_test)}
Path(METRICS_PATH).write_text(json.dumps(metrics, indent=2))
print(metrics)


{'mae': 97036.93469137883, 'rmse': np.float64(253291.7181513605), 'r2': 0.46657868405065117, 'mdape_percent': np.float64(18.845819548431844), 'baseline_median_mae': 171493.008766427, 'n_train': 607585, 'n_test': 186507}


In [22]:
sample = X_test.iloc[[0]]
pred_gbp = np.expm1(pipe.predict(sample))[0]
actual_gbp = np.expm1(y_test.iloc[0])
print(pred_gbp, actual_gbp)


304000.49052411783 445999.9999999996
